<a href="https://colab.research.google.com/github/brandonwolk/kaneohe-coral-mcda/blob/main/Physcial_Varibales_8_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL 1: CONFIG_AND_SETUP]
import os
import rasterio
import geopandas as gpd
import numpy as np
from rasterio.fill import fillnodata
from shapely.geometry import box
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define Project Paths
BASE_DIR = '/content/drive/MyDrive/Capstone'
RAW_DATA_DIR = os.path.join(BASE_DIR, '01 Raw Data')
PROCESSED_DIR = os.path.join(BASE_DIR, '02 Processed Rasters')

# Ensure processed directory exists
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Define Kaneohe Bay Bounding Box (Lat/Long)
BBOX = (-157.86, 21.40, -157.73, 21.54)
print("Environment configured and directories verified.")

Mounted at /content/drive
Environment configured and directories verified.


In [ ]:
# [CELL 2: BATHYMETRY_PROCESSING (Updated)]
def process_bathymetry():
    in_path = os.path.join(RAW_DATA_DIR, 'kaneohe_MB_Lidar_4m.asc')
    out_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Bathy_Final.tif')

    with rasterio.open(in_path) as src:
        print(f"Native Shape: {src.shape}")
        data = src.read(1)

        # Apply depth constraints (-100m to 2m) to prevent land-bleed during interpolation
        data_to_fill = np.where((data > -100) & (data < 2), data, -9999)

        # Perform masked fillnodata gap-filling
        filled = fillnodata(data_to_fill, mask=(data_to_fill != -9999), max_search_distance=10)

        # Build clean profile and assign EPSG:6634 (NAD83(PA11) UTM Zone 4N - Hawaii LiDAR standard)
        profile = src.profile.copy()
        profile.update(
            driver='GTiff',
            nodata=-9999,
            crs='EPSG:6634'
        )

        # Remove conflicting block keys to eliminate the CPLE_IllegalArg warning
        for key in ['BLOCKXSIZE', 'BLOCKYSIZE', 'TILED']:
            profile.pop(key, None)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(filled, 1)

    print(f"Bathymetry processing complete with assigned CRS (EPSG:6634). Saved to: {out_path}")

# Run the updated function
process_bathymetry()

Native Shape: (3853, 3531)


Bathymetry processing complete with assigned CRS (EPSG:6634). Saved to: /content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Bathy_Final.tif


In [ ]:
# [CELL 3: BENTHIC_PROCESSING]
def process_benthic():
    in_path = os.path.join(RAW_DATA_DIR, 'hi_noaa_oahu_benthic_habitats.zip')
    out_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Benthic_GIS.geojson')

    # Read zipped vector data via GeoPandas
    gdf = gpd.read_file(f"zip://{in_path}")
    print(f"Benthic Native CRS: {gdf.crs}")
    print(f"Benthic Initial Feature Count: {len(gdf)}")

    # Define Kaneohe Bay bounding box in WGS84 (EPSG:4326)
    bbox_geom = box(-157.86, 21.40, -157.73, 21.54)
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

    # Reproject benthic data to match bathymetry CRS (EPSG:6634) and project bounding box accordingly
    gdf_proj = gdf.to_crs("EPSG:6634")
    bbox_proj = bbox_gdf.to_crs("EPSG:6634")

    # Spatial clip to Kaneohe Bay extent
    clipped = gpd.clip(gdf_proj, bbox_proj)

    print(f"Clipped & Projected Feature Count: {len(clipped)}")
    print(f"Final Vector CRS: {clipped.crs}")

    # Export to GeoJSON for seamless ArcGIS Pro integration
    clipped.to_file(out_path, driver="GeoJSON")
    print(f"Benthic processing complete. Saved to: {out_path}")

# Run the benthic processing function
process_benthic()

Benthic Native CRS: EPSG:4326
Benthic Initial Feature Count: 2380
Clipped & Projected Feature Count: 531
Final Vector CRS: EPSG:6634
Benthic processing complete. Saved to: /content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Benthic_GIS.geojson


In [ ]:
# [CELL: EXPORT_AS_SHAPEFILE]
import os
import geopandas as gpd

# Load the processed benthic data we already clipped
benthic_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Benthic_GIS.geojson')
gdf_benthic = gpd.read_file(benthic_path)

# Export as an ESRI Shapefile (creates a set of files ending in .shp, .shx, .dbf, etc.)
shp_out_path = os.path.join(PROCESSED_DIR, 'Kaneohe_Benthic.shp')
gdf_benthic.to_file(shp_out_path, driver='ESRI Shapefile')

print(f"Shapefile successfully created at: {shp_out_path}")

Shapefile successfully created at: /content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Benthic.shp


In [ ]:
# [CELL: DOWNLOAD_CDIP_WAVE_DATA]
import os
import pandas as pd

# Your exact ERDDAP URL for Station 198 (2000-2025 baseline)
erddap_url = "https://erddap.cdip.ucsd.edu/erddap/tabledap/wave_agg.csv?station_id%2Ctime%2CwaveHs%2CwaveTp%2CwaveTa%2CwaveDp%2CmetaStationName%2Clatitude%2Clongitude&station_id=%22198%22&time%3E2000-01-01T00%3A00%3A00Z&time%3C=2025-12-31T23%3A59%3A59Z&waveFlagPrimary=1"

# Define target folder in your Google Drive
raw_dir = '/content/drive/MyDrive/Capstone/01 Raw Data'
os.makedirs(raw_dir, exist_ok=True)
out_path = os.path.join(raw_dir, 'CDIP_Station198_WaveData.csv')

print("Fetching 25 years of wave data directly from CDIP server...")
df_wave = pd.read_csv(erddap_url)

# Save straight into your Drive
df_wave.to_csv(out_path, index=False)

print(f"Success! Saved directly to your Drive at: {out_path}")
print(f"Total historical wave records pulled: {len(df_wave):,}")
print(df_wave.head())

Fetching 25 years of wave data directly from CDIP server...


/tmp/ipykernel_2847/3741452223.py:14: DtypeWarning: Columns (2,3,4,5,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wave = pd.read_csv(erddap_url)


Success! Saved directly to your Drive at: /content/drive/MyDrive/Capstone/01 Raw Data/CDIP_Station198_WaveData.csv
Total historical wave records pulled: 2,199,385
   station_id                  time waveHs     waveTp    waveTa   waveDp  \
0         NaN                   UTC  meter     second    second  degreeT   
1       198.0  2012-10-26T23:12:07Z   0.83  14.285714  6.768145  352.125   
2       198.0  2012-10-26T23:12:07Z   0.83  14.285714  6.768145  352.125   
3       198.0  2012-10-26T23:12:07Z   0.83  14.285714  6.768145  352.125   
4       198.0  2012-10-26T23:12:07Z   0.83  14.285714  6.768145  352.125   

                metaStationName       latitude     longitude  
0                           NaN  degrees_north  degrees_east  
1  KANEOHE BAY, HI BUOY - 198p1       21.47747    -157.75262  
2  KANEOHE BAY, HI BUOY - 198p1       21.47747    -157.75262  
3  KANEOHE BAY, HI BUOY - 198p1       21.47747    -157.75262  
4  KANEOHE BAY, HI BUOY - 198p1       21.47747    -157.75262  


In [ ]:
# [CELL: CALCULATE_CDIP_WAVE_STATS]
import pandas as pd
import numpy as np

# Path to the file we just saved in your Drive
path = '/content/drive/MyDrive/Capstone/01 Raw Data/CDIP_Station198_WaveData.csv'

# Load CSV and skip the second row (which contains the unit labels like 'meter')
df_wave = pd.read_csv(path, skiprows=[1])

# Convert wave height (waveHs) to numeric, turning any text errors into NaN
df_wave['waveHs'] = pd.to_numeric(df_wave['waveHs'], errors='coerce')
wave_heights = df_wave['waveHs'].dropna()

# Calculate empirical baseline metrics
mean_hs = wave_heights.mean()
median_hs = wave_heights.median()
p95_hs = np.percentile(wave_heights, 95)
p99_hs = np.percentile(wave_heights, 99)
max_hs = wave_heights.max()

print("--- Kaneohe Bay Station 198 Wave Climatology (2000-2025) ---")
print(f"Total Valid Records Analyzed: {len(wave_heights):,}")
print(f"Mean Significant Wave Height (Mean Hs): {mean_hs:.2f} meters")
print(f"Median Significant Wave Height: {median_hs:.2f} meters")
print(f"95th Percentile Wave Height: {p95_hs:.2f} meters")
print(f"99th Percentile Storm Threshold (99th Pct): {p99_hs:.2f} meters")
print(f"Maximum Recorded Storm Wave Height: {max_hs:.2f} meters")

--- Kaneohe Bay Station 198 Wave Climatology (2000-2025) ---
Total Valid Records Analyzed: 2,199,384
Mean Significant Wave Height (Mean Hs): 1.69 meters
Median Significant Wave Height: 1.59 meters
95th Percentile Wave Height: 2.75 meters
99th Percentile Storm Threshold (99th Pct): 3.44 meters
Maximum Recorded Storm Wave Height: 5.15 meters


ModuleNotFoundError: No module named 'rioxarray'